In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

import datetime

import pandas as pd


from selenium import webdriver


from time import sleep

import os

from selenium.webdriver.common.by import By

from bs4 import BeautifulSoup

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# %%

In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'CR SUGEVAL' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

os.chdir(scriptfolder)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process


if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)

Running CR SUGEVAL Web Scraping Tool v.1.0


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()

# %%

In [4]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

        regulatorName+' 1': 'https://aplicaciones.sugeval.fi.cr/Participantes/AuditoresExternos',
        regulatorName+' 2': 'https://aplicaciones.sugeval.fi.cr/Participantes/Calificadoras',
        regulatorName+' 3': 'https://aplicaciones.sugeval.fi.cr/Participantes/Custodios',
        regulatorName+' 4': 'https://aplicaciones.sugeval.fi.cr/Participantes/Emisores',
        regulatorName+' 5': 'https://aplicaciones.sugeval.fi.cr/Participantes/GruposFinancieros',
        regulatorName+' 6': 'https://aplicaciones.sugeval.fi.cr/Participantes/ProveedoresPrecios',        
        regulatorName+' 7': 'https://aplicaciones.sugeval.fi.cr/Participantes/PuestosBolsa',
        regulatorName+' 8': 'https://aplicaciones.sugeval.fi.cr/Participantes/SAFI',
        regulatorName+' 9': 'https://aplicaciones.sugeval.fi.cr/Participantes/OtrosParticipantes',    

        }



Typology={

        regulatorName+' 1':    'Auditores externos',
        regulatorName+' 2':    'Calificadoras de riesgo',
        regulatorName+' 3':    'Custodios',
        regulatorName+' 4':    'Emisores',
        regulatorName+' 5':    'Grupos financieros',
        regulatorName+' 6':   'Proveedor de precios',
        regulatorName+' 7':    'Puestos de bolsa',
        regulatorName+' 8':    'Sociedades administradoras de fondos',
        regulatorName+' 9':   'Otros participantes'

        }



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [],
        }


now = datetime.datetime.now()

processdate = now.strftime('%Y-%m-%d')




In [5]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


In [ ]:

# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):

    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")

    driver.get(regdict[reg])   
    
    wait = WebDriverWait(driver, 20)  # adjust timeout as needed

    btn_element = wait.until(EC.element_to_be_clickable((By.ID, "btnExportar")))
    btn_element.click()

    sleep(10)

    dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
    if len(dl_files)>0 and not dl_files[0].endswith('.tmp') and not dl_files[0].endswith('.crdownload'):
        print('[INFO] -- Check the download file --')
        sleep(3)

    else:
        print('[INFO] -- Maybe the file link is still downloading -- ')
        sleep(30)
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]

    with pd.ExcelFile(dl_files[0]) as xlsx:
        print(f"[INFO] -- Number of sheets: {len(xlsx.sheet_names)}  --")
        print(f"[INFO] -- Number of sheets: {xlsx.sheet_names}  --") 
        data = pd.read_excel(xlsx, sheet_name=xlsx.sheet_names[0])

        data = data.dropna(subset=[data.columns[0], data.columns[1]])
        data.columns = data.iloc[0]
        data = data[1:].reset_index(drop=True)
        print(data.shape)
        # if reg != regulatorName + ' 9':
        for _,item_ in data.iterrows():
            name_ = item_[data.columns[0]]
            #regulador = item_[data.columns[-1]]
            sqldict['Name'].append(name_)
            sqldict['RegulationType'].append('Regulated')
            sqldict['ListProcessDate'].append(processdate)
            sqldict['ListName'].append('Registro Nacional de Valores e Intermediarios')
            sqldict['Typology'].append(Typology[reg])
            sqldict['RegCtry'].append(reg.split(' ')[0])
            sqldict['RegCode'].append(reg.split(' ')[1])
            sqldict['ListCode'].append('1')
            sqldict = bourange_same_length_array(sqldict)
        # else:  


    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))

[INFO] : Working 1/9 _(CR SUGEVAL 1)_ 


[INFO] -- Check the download file --
[INFO] -- Number of sheets: 1  --
[INFO] -- Number of sheets: ['Resultado']  --
(66, 3)
[INFO] : Working 2/9 _(CR SUGEVAL 2)_ 
[INFO] -- Check the download file --
[INFO] -- Number of sheets: 1  --
[INFO] -- Number of sheets: ['Resultado']  --
(4, 3)
[INFO] : Working 3/9 _(CR SUGEVAL 3)_ 
[INFO] -- Check the download file --
[INFO] -- Number of sheets: 1  --
[INFO] -- Number of sheets: ['Resultado']  --
(22, 3)
[INFO] : Working 4/9 _(CR SUGEVAL 4)_ 
[INFO] -- Check the download file --
[INFO] -- Number of sheets: 1  --
[INFO] -- Number of sheets: ['Resultado']  --
(48, 3)
[INFO] : Working 5/9 _(CR SUGEVAL 5)_ 
[INFO] -- Check the download file --
[INFO] -- Number of sheets: 1  --
[INFO] -- Number of sheets: ['Resultado']  --
(4, 3)
[INFO] : Working 6/9 _(CR SUGEVAL 6)_ 
[INFO] -- Check the download file --
[INFO] -- Number of sheets: 1  --
[INFO] -- Number of sheets: ['Resultado']  --
(2, 3)
[INFO] : Working 7/9 _(CR SUGEVAL 7)_ 
[INFO] -- Check the

In [7]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(filename,index=False)



driver.quit()

sleep(3)

